In [1]:
import pandas as pd
from top2vec import Top2Vec
import os
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import umap
import plotly.express as px
from pprint import pprint

/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/only-kkk-revival.feather')

In [3]:
data.shape

(53598, 6)

In [4]:
stopwords=stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [5]:
data['article_stop'] = data['article'].str.lower().str.split().apply(lambda x: [word for word in x if word not in stopwords])

In [6]:
data['article_lemma'] = data['article_stop'].apply(lambda x: [WordNetLemmatizer().lemmatize(word) for word in x])

In [7]:
data['article_lemma_string']=data['article_lemma'].apply(lambda x: ' '.join(x))

In [8]:
Counter(data['article_lemma'].explode()).most_common(10)

[('ku', 55013),
 ('klux', 51841),
 ('state', 43756),
 ('khan', 37339),
 ('klan', 36358),
 ('one', 27774),
 ('would', 27433),
 ('said', 25386),
 ('new', 24868),
 ('|', 24645)]

In [9]:
model=Top2Vec(documents=data['article_lemma_string'].tolist(), speed="learn", workers=4, embedding_model='distiluse-base-multilingual-cased')

2025-11-21 18:51:52,698 - top2vec - INFO - Pre-processing documents for training
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2025-11-21 18:52:05,381 - top2vec - INFO - Downloading distiluse-base-multilingual-cased model
2025-11-21 18:52:06,879 - top2vec - INFO - Creating joint document/word embedding
2025-11-21 18:53:35,804 - top2vec - INFO - Creating lower dimension embedding of documents
2025-11-21 18:53:52,363 - top2vec - INFO - Finding dense areas of documents
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed 

In [10]:
topic_sizes, topic_nums = model.get_topic_sizes()

In [11]:
print(len(topic_sizes), len(topic_nums)) #955 topics with distiluse-base-multilingual-cased

278 278


In [12]:
id_dic={}
topic_id={}
for element in zip(topic_nums, topic_sizes):
    documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=element[0], num_docs=element[1])
    for score, id in zip(document_scores, document_ids):
        id_dic[id]=score
        topic_id[id]=element[0]

In [13]:
topic_words, word_scores, topic_scores = model.get_topics(len(topic_sizes))

In [14]:
df_words=pd.DataFrame(topic_words).transpose()
df_words.columns=topic_nums
df_words.columns = df_words.columns.astype(str)

In [28]:
df_words.to_csv('/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/only-kkk-topic-words.csv', index=False)

In [15]:
data['topic_id']=data.index.map(topic_id) #map topic id to each document
data['topic_score']=data.index.map(id_dic) #map topic score to each document

In [20]:
data['vector']=model.document_vectors.tolist()

In [16]:
df_words[['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15']].iloc[:10]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,electorate,khan,catholicism,racial,parade,judicial,legislature,prosecution,letter,senate,judiciary,policemen,theatrical,senate,tw,khan
1,election,emperor,catholic,negroes,parades,prosecution,impeached,acquitted,envelope,indianapolis,senate,police,dramatic,congressional,ow,cherokee
2,electoral,dictatorship,catholics,negro,ceremony,litigation,governor,jailed,letters,republican,judicial,policeman,theatre,legislature,hawk,kiux
3,caucus,clancy,protestant,blacks,ceremonial,tribunal,senate,judicial,mailed,senatorial,courthouse,arrests,theater,congressman,fh,klux
4,senate,caste,church,nigger,festival,jury,legislative,convicted,handwriting,senator,negro,kidnapped,drama,congress,ha,cork
5,elections,corruption,episcopal,black,ceremonies,judiciary,impeachment,prosecuted,mailing,caucus,impeached,arrest,scenes,legislative,wss,klon
6,reelected,corrup,protestants,blackwell,crowded,juris,governors,jail,mail,republicans,court,arrested,history,senatorial,pow,kians
7,elected,constitution,churches,slavery,crowd,court,gubernatorial,prosecute,writes,indian,courts,arresting,circus,parliamentary,huff,kendrick
8,suffrage,clan,vatican,slaves,protested,acquitted,congressional,prosecuting,petition,congressman,tribunal,sheriffs,historical,legislators,hf,kieagle
9,legislature,kians,crusade,caucasian,protesting,judge,impeach,acquittal,unwritten,congressional,congressman,cop,historic,senator,hs,quail


In [18]:
pprint(data[data['topic_id']==0]['article'].iloc[10])  #example article from topic 0

('Sergeant Eatcs was on the streets several\n'
 'emtion ] he day. he was naiea npon by\n'
 'q large number OF prominent citizens. He will\n'
 'leave ; i2r ,WaSshinGt%m ut A. NJ tomorrow\n'
 'gong VIA frcdcr!ckSburE. and his present plan\n'
 'is to stop while AZ Mount Vernon. the home\n'
 'OF Washington.\n'
 '\n'
 '\n'
 'In the convention Yesterday, E. Gibson called\n'
 'dicr OF the Union, who WAS marching through\n'
 "the South ':avnE the United States flag and\n"
 'ing. was in the city, and moved that he be In\n'
 'vitcd TO the floor OF the convention Clements\n'
 '1h2dhcaD moved to lay the motion on the table,\n'
 "when mo'lon was carried by vote OF 4S to 25.\n"
 'an inc ooHServaiivcs voted against the motion\n'
 'invite Inc sergeant If they had had the chance.\n'
 '\n'
 '\n'
 'pr; John Spotswood Wexford has been chosen\n'
 '10 ~ me chair OF materia medica and thera-\n'
 'CO. ed by the resignation of Dr. b. r. Wexford.\n'
 'Ihe later was hcnorcd by the board with the\n'
 'degree or e

In [23]:
embedding = umap.UMAP().fit_transform(data['vector'].tolist()) #reduce dimensionality of vector representation

In [24]:
clusterable_embedding = umap.UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=3,
    random_state=42,
).fit_transform(embedding.data)

/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [33]:
fig = px.scatter_3d(
    data, 
    x=clusterable_embedding[:, 0], 
    y=clusterable_embedding[:, 1], 
    z=clusterable_embedding[:, 2], 
    color='topic_id',  # Specify the column for color
    hover_data=['topic_id']
)

fig.update_layout(
    autosize=False,
    width=1000,
    height=1000,
    title={"text": "The Vector Representation of KKK Articles with UMAP Dimension Reduction",
            "x": 0.5,
            "xanchor": "center"},
    scene=dict(
        aspectmode="manual",
        aspectratio=dict(x=1.3, y=1.3, z=1.3)  # ⬅ bigger plotting box
    ),
    scene_camera=dict(
        eye=dict(x=2, y=2, z=2)  # zoom out view
    )
)

fig.update_traces(marker=dict(size=3))

# fig.show()
fig.write_html("/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/interactive-plot-3d-new.html")
fig.write_image("/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/umap-3d-new.png", scale=4)

In [34]:
from PIL import Image
Image.open("/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/umap-3d-new.png").save("/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/umap-3d-new.pdf")

In [38]:
data["sn"]=data["index"].str.extract(r"(sn[0-9A-Za-z]+)")

In [39]:
data.to_feather('/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/only-kkk-revival-top2vec.feather')
data.to_csv('/Volumes/T7/chroniclingamerica/american-stories/nlp4dh/only-kkk/only-kkk-revival-top2vec.csv', index=False)

In [40]:
data['sn']

0        sn92070146
1        sn83045211
2        sn85038485
3        sn85026941
4        sn86063782
            ...    
53593    sn82014085
53594    sn83045499
53595    sn83045324
53596    sn83045324
53597    sn83045863
Name: sn, Length: 53598, dtype: object